<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 1 — Pipeline de preprocesamiento de texto</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">De texto crudo a términos limpios · NLTK + spaCy (es_core_news_sm)</div></div>

## Reglas de entrega

- **Repo:** suban este notebook (ya ejecutado, con sus salidas) a su repositorio de GitHub Classroom de la organización `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio:** si usaron IA en cualquier parte, declárenlo en ese archivo (qué herramienta, para qué celda, qué les dio y qué cambiaron). No declararlo se considera falta de honestidad académica.
- **Defensa oral (eliminatoria):** se les preguntará por *cualquier* celda. "Lo generó la IA" o "lo dijo el profesor" no son justificaciones válidas. Si no pueden explicar su código, no hay calificación.
- **Penalizaciones por entrega tardía:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Las celdas marcadas con `# TODO` y las preguntas en **negritas** son lo que se evalúa. El resto es andamiaje que ya viene resuelto para que enfoquen su tiempo en lo que importa.

## Objetivo

Construir el pipeline que convierte una noticia cruda en una lista de términos limpios, tomando **decisiones justificadas** en cada paso (no recetas memorizadas). El corpus que limpien aquí es el mismo que usarán en el **Lab 2** para construir su motor de búsqueda, así que las decisiones de hoy se arrastran al resto de la unidad.

## 0 · Entorno

Ejecuten una sola vez. Si están en Colab, descomenten la primera línea.

In [9]:
# !pip install -q nltk spacy && python -m spacy download es_core_news_sm
import re, unicodedata, json
from collections import Counter
import pandas as pd

import nltk
nltk.download('punkt', quiet=True)
from nltk.stem import SnowballStemmer

import spacy
try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    from spacy.cli import download
    download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')

print('Entorno listo.')

Entorno listo.


## Corpus

Noticias chiapanecas (sintéticas) con ruido real: HTML, URLs, emojis, acentos faltantes, dobles espacios. **No editen el corpus.**

In [10]:
corpus_crudo = [
    {"id": "d01", "titulo": "Lluvias provocan inundaciones en Tuxtla",
     "texto": "Las  fuertes lluvias   provocaron inundaciones en varias colonias del sur de Tuxtla Gutierrez 😟. "
              "Proteccion Civil pidio a la poblacion no cruzar las calles anegadas. Mas info en https://chiapasparalelo.com/nota1 ."},
    {"id": "d02", "titulo": "Crisis hidrica golpea la region",
     "texto": "La crisis hidrica se agrava: el desabasto del liquido vital afecta a miles de familias en la zona alta. "
              "Las autoridades atribuyen la escasez a la prolongada sequia y a la falta de mantenimiento de los pozos."},
    {"id": "d03", "titulo": "Cafe de Chiapas rompe record de exportacion",
     "texto": "<p>El <b>cafe</b> de Chiapas rompio su record historico de exportacion este ciclo, impulsado por la demanda en Europa y Asia.</p> "
              "Los productores de la Sierra celebran precios al alza."},
    {"id": "d04", "titulo": "Sequia afecta cultivos de maiz",
     "texto": "La sequia afecta gravemente los cultivos de maiz y frijol en la region fronteriza. "
              "Los agricultores reportan perdidas de hasta el 40% y piden apoyos al gobierno estatal."},
    {"id": "d05", "titulo": "Turismo crece en el Canon del Sumidero",
     "texto": "El Canon del Sumidero recibio mas de 200 mil visitantes durante la temporada. "
              "Los recorridos en lancha y el avistamiento de fauna son los principales atractivos del parque nacional. #Chiapas"},
    {"id": "d06", "titulo": "Sismo de magnitud 5.1 frente a las costas",
     "texto": "Un sismo de magnitud 5.1 se registro frente a las costas de Chiapas la madrugada del martes. "
              "No se reportaron danos ni victimas, informo el Servicio Sismologico Nacional."},
    {"id": "d07", "titulo": "UPCh inaugura laboratorio de IA",
     "texto": "La Universidad Politecnica de Chiapas inauguro un nuevo laboratorio de inteligencia artificial "
              "equipado con GPUs para proyectos de aprendizaje automatico y vision por computadora. Visita https://upchiapas.edu.mx ."},
    {"id": "d08", "titulo": "Repunta la produccion de cacao",
     "texto": "La produccion de cacao en la region del Soconusco repunto este año tras varios ciclos a la baja. "
              "Cooperativas locales apuestan por el chocolate artesanal de origen para mercados premium."},
    {"id": "d09", "titulo": "San Cristobal, destino cultural",
     "texto": "San Cristobal de las Casas se consolida como destino cultural: sus mercados, iglesias y cafeterias "
              "atraen a viajeros de todo el mundo. La gastronomia y el textil artesanal son protagonistas."},
    {"id": "d10", "titulo": "Avanza obra de infraestructura carretera",
     "texto": "Avanza la rehabilitacion de la carretera que conecta Tuxtla con la costa. "
              "La obra busca reducir tiempos de traslado y mejorar la seguridad vial para miles de automovilistas."},
    {"id": "d11", "titulo": "Alertan por casos de dengue",
     "texto": "La Secretaria de Salud alerto por un repunte de casos de dengue en municipios de tierra caliente. "
              "Pide a la poblacion eliminar criaderos de mosco y acudir al medico ante fiebre alta. 🦟"},
    {"id": "d12", "titulo": "Feria celebra el cafe y el cacao",
     "texto": "La feria regional celebro el cafe y el cacao chiapaneco con catas, musica y venta directa de productores. "
              "Miles de asistentes recorrieron los stands durante el fin de semana."},
    {"id": "d13", "titulo": "Restablecen servicio de agua potable",
     "texto": "El servicio de agua potable se restablecera de forma escalonada en las colonias afectadas por la falla en la red. "
              "El organismo operador pidio a los usuarios almacenar agua de manera responsable."},
    {"id": "d14", "titulo": "Estudiantes ganan concurso de robotica",
     "texto": "Estudiantes de ingenieria de Tuxtla ganaron el primer lugar en un concurso nacional de robotica 🤖 "
              "con un brazo robotico de bajo costo. El equipo representara a Mexico en una competencia internacional."},
]
print(f"{len(corpus_crudo)} documentos cargados.")

14 documentos cargados.


In [11]:
df = pd.DataFrame(corpus_crudo)
df['n_chars'] = df['texto'].str.len()
df[['id', 'titulo', 'n_chars']]

,id,titulo,n_chars
0,d01,Lluvias provocan inundaciones en Tuxtla,213
1,d02,Crisis hidrica golpea la region,207
2,d03,Cafe de Chiapas rompe record de exportacion,184
3,d04,Sequia afecta cultivos de maiz,169
4,d05,Turismo crece en el Canon del Sumidero,190
5,d06,Sismo de magnitud 5.1 frente a las costas,170
6,d07,UPCh inaugura laboratorio de IA,213
7,d08,Repunta la produccion de cacao,186
8,d09,"San Cristobal, destino cultural",190
9,d10,Avanza obra de infraestructura carretera,173


## 1 · Cargar y explorar

Antes de limpiar, hay que **mirar** los datos. Una limpieza a ciegas destruye señal sin que se den cuenta.

**1.a** Estadísticas de longitud. Completen los cálculos.

In [12]:
# TODO: reporten el numero de documentos, la longitud media en caracteres,
#       y la longitud media en PALABRAS (split ingenuo por espacios).
#       Pista: df['texto'].str.split().str.len()

n_docs = len(corpus_crudo)
longitud_chars = [len(doc['texto']) for doc in corpus_crudo]
longitud_palabras = [len(doc['texto'].split()) for doc in corpus_crudo]

avg_chars = sum(longitud_chars) / n_docs
avg_palabras = sum(longitud_palabras) / n_docs

print(f"Número de documentos: {n_docs}")
print(f"Longitud media en caracteres: {avg_chars:.2f}")
print(f"Longitud media en palabras: {avg_palabras:.2f}")

Número de documentos: 14
Longitud media en caracteres: 189.07
Longitud media en palabras: 30.21


**1.b** Detección de ruido. Les damos un detector de URLs ya hecho como ejemplo. **Completen** los detectores de etiquetas HTML y de emojis, y reporten en qué documentos aparece cada tipo de ruido.

In [13]:
RE_URL  = re.compile(r'https?://\S+')
RE_HTML = re.compile(r'<[^>]+>')  # TODO: regex que capture etiquetas como <p> </b>
RE_EMOJI = re.compile(r'[\U0001F300-\U0001F9FF]|[\U0001F600-\U0001F64F]|[\U0001F680-\U0001F6FF]|[😀-🤯🦀-🦟]', flags=re.UNICODE)  # TODO: regex para detectar emojis

for fila in corpus_crudo:
    urls = RE_URL.findall(fila['texto'])
    html_tags = RE_HTML.findall(fila['texto'])
    emojis = RE_EMOJI.findall(fila['texto'])
    
    if urls:
        print(fila['id'], '-> URL:', urls)
    if html_tags:
        print(fila['id'], '-> HTML:', html_tags)
    if emojis:
        print(fila['id'], '-> EMOJI:', emojis)

d01 -> URL: ['https://chiapasparalelo.com/nota1']
d01 -> EMOJI: ['😟']
d03 -> HTML: ['<p>', '<b>', '</b>', '</p>']
d07 -> URL: ['https://upchiapas.edu.mx']
d11 -> EMOJI: ['🦟']
d14 -> EMOJI: ['🤖']


> **Pregunta (defensa):** de los tres tipos de ruido, ¿cuál *podría* ser señal útil en algún dominio y por qué? Respondan en la celda de abajo.

_Su respuesta:_ Los **emojis** podrían ser señal útil en un clasificador de sentimientos (😟 = preocupación, 🤖 = tecnología). Sin embargo, en nuestro motor de búsqueda enfocado en términos limpios, eliminamos emojis porque la información emocional que comunican ya está codificada en palabras del texto (ej: 'inundación', 'alertan', 'crisis'). Las URLs y tags HTML son siempre ruido: son metadatos de distribución/formato, nunca semántica.

## 2 · Normalización

Primer paso del pipeline: limpiar caracteres especiales, acentos, espacios.

In [14]:
def normalizar(texto):
    """Convierte el texto a minúsculas, normaliza unicode (quita acentos) y elimina ruido."""
    # TODO: normalizar acentos, minusculas, quitar URLs, HTML, emojis, hashtags, espacios multiples
    # Convertir a minúsculas
    texto = texto.lower()
    
    # Normalizar unicode: descomponer caracteres acentuados (NFD = decomposed form)
    texto_nfd = unicodedata.normalize('NFD', texto)
    # Filtrar solo caracteres base (quitar marcas diacríticas con categoria 'Mn')
    texto_sin_acentos = ''.join(c for c in texto_nfd if unicodedata.category(c) != 'Mn')
    
    # Quitar URLs
    texto_sin_acentos = RE_URL.sub('', texto_sin_acentos)
    
    # Quitar HTML tags
    texto_sin_acentos = RE_HTML.sub('', texto_sin_acentos)
    
    # Quitar emojis
    texto_sin_acentos = RE_EMOJI.sub('', texto_sin_acentos)
    
    # Quitar hashtags
    texto_sin_acentos = re.sub(r'#\s*', '', texto_sin_acentos)
    
    # Colapsar múltiples espacios en blanco
    texto_sin_acentos = re.sub(r'\s+', ' ', texto_sin_acentos).strip()
    
    return texto_sin_acentos

# Demostración
ejemplo = corpus_crudo[0]['texto']
print("Original:", ejemplo[:100] + "...")
print("Normalizado:", normalizar(ejemplo)[:100] + "...")

Original: Las  fuertes lluvias   provocaron inundaciones en varias colonias del sur de Tuxtla Gutierrez 😟. Pro...
Normalizado: las fuertes lluvias provocaron inundaciones en varias colonias del sur de tuxtla gutierrez . protecc...


## 3 · Stopwords

Palabras muy frecuentes que no aportan significado. Pero cuidado: a veces SÍ aportan.

In [15]:
stopwords_es = nlp.Defaults.stop_words
print('Total de stopwords de spaCy (es):', len(stopwords_es))
print(sorted(list(stopwords_es))[:30])

Total de stopwords de spaCy (es): 521
['a', 'acuerdo', 'adelante', 'ademas', 'además', 'afirmó', 'agregó', 'ahi', 'ahora', 'ahí', 'al', 'algo', 'alguna', 'algunas', 'alguno', 'algunos', 'algún', 'alli', 'allí', 'alrededor', 'ambos', 'ante', 'anterior', 'antes', 'apenas', 'aproximadamente', 'aquel', 'aquella', 'aquellas', 'aquello']


**3.a** Inspeccionen la lista y encuentren **al menos una** palabra que, en el dominio de noticias de Chiapas, *sí* podría ser señal (piensen en negaciones, intensificadores o términos que cambian el sentido). Construyan su propia lista `MIS_STOPWORDS` quitándola(s).

In [16]:
CONSERVAR = {'no', 'muy', 'mas'}   # TODO: palabras que NO quieren tratar como stopword (con criterio de dominio)
MIS_STOPWORDS = set(stopwords_es) - CONSERVAR

# TODO: impriman cuantas quitaron y verifiquen que su decision tiene efecto
print(f"Palabras conservadas: {CONSERVAR}")
print(f"Stopwords originales de spaCy: {len(stopwords_es)}")
print(f"Stopwords en MIS_STOPWORDS: {len(MIS_STOPWORDS)}")
print(f"Total de palabras quitadas de la lista de stopwords: {len(CONSERVAR)}")

Palabras conservadas: {'mas', 'no', 'muy'}
Stopwords originales de spaCy: 521
Stopwords en MIS_STOPWORDS: 518
Total de palabras quitadas de la lista de stopwords: 3


_Justifiquen qué conservaron y por qué (lo defenderán oralmente):_

_respuesta:_

Conservamos **{'no', 'muy', 'mas'}** porque son palabras clave en dominio de noticias/crisis:

1. **'no'** (negación): 'no afecta cultivos' ≠ 'afecta cultivos' → significado opuesto. Es crítico en reportes de salud/desastres.
2. **'muy'** (intensificador): 'afecta gravemente' < 'afecta muy gravemente' → matiza severidad. Importante en crisis.
3. **'mas'** (comparativo/arcaísmo): 'quieren mas apoyo' ≠ 'quieren mas (pero no lo tienen)' → preserva ambigüedad.

En dominio de noticias chiapanecas sobre crisis, estas palabras aportan información crítica que se pierde si se eliminan.

## 4 · Stemming vs. lemmatización

Mismo objetivo, dos filosofías. Aquí **miden** la diferencia en vez de creerla.

In [17]:
stemmer = SnowballStemmer('spanish')

def tokens_stemming(texto):
    # TODO: normalizar -> tokenizar -> quitar stopwords/puntuacion -> aplicar stemmer
    texto_norm = normalizar(texto)
    palabras = texto_norm.split()
    tokens = [p for p in palabras if p not in MIS_STOPWORDS and p.isalnum()]
    tokens_stem = [stemmer.stem(t) for t in tokens]
    return tokens_stem

def tokens_lemma(texto):
    # TODO: normalizar -> procesar con spaCy -> token.lemma_ -> quitar stopwords/puntuacion
    texto_norm = normalizar(texto)
    doc = nlp(texto_norm)
    tokens_lemma_list = [token.lemma_ for token in doc if token.lemma_ not in MIS_STOPWORDS and token.is_alpha]
    return tokens_lemma_list

# Demostración rápida
print("Ejemplo con stemming:", tokens_stemming(corpus_crudo[0]['texto'])[:8])
print("Ejemplo con lemma:", tokens_lemma(corpus_crudo[0]['texto'])[:8])

Ejemplo con stemming: ['fuert', 'lluvi', 'provoc', 'inund', 'coloni', 'sur', 'tuxtl', 'gutierrez']
Ejemplo con lemma: ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez']


**4.a** Apliquen ambos a todo el corpus y comparen el **tamaño del vocabulario** resultante.

In [18]:
# TODO: construir el vocabulario (set de terminos) con cada metodo sobre TODO el corpus
#       y reportar |V_stemming| vs |V_lemma|. Muestren 10 ejemplos donde difieran.
print("Construyendo vocabularios (esto puede tomar ~10s)...\n")

vocab_stemming = set()
vocab_lemma = set()

for doc in corpus_crudo:
    vocab_stemming.update(tokens_stemming(doc['texto']))
    vocab_lemma.update(tokens_lemma(doc['texto']))

print(f"Tamaño vocabulario (Stemming): {len(vocab_stemming)} términos")
print(f"Tamaño vocabulario (Lemmatización): {len(vocab_lemma)} términos")
print(f"Diferencia: {abs(len(vocab_stemming) - len(vocab_lemma))} términos\n")

# Encontrar ejemplos donde difieran
solo_en_stemming = vocab_stemming - vocab_lemma
solo_en_lemma = vocab_lemma - vocab_stemming

print("Ejemplos de términos SOLO en STEMMING (no en LEMMA):")
print(list(solo_en_stemming)[:10])

print("\nEjemplos de términos SOLO en LEMMA (no en STEMMING):")
print(list(solo_en_lemma)[:10])

Construyendo vocabularios (esto puede tomar ~10s)...

Tamaño vocabulario (Stemming): 167 términos
Tamaño vocabulario (Lemmatización): 199 términos
Diferencia: 32 términos

Ejemplos de términos SOLO en STEMMING (no en LEMMA):
['chiap', 'lug', 'tiemp', 'dan', 'viajer', 'cicl', 'histor', 'madrug', 'sumider', 'alert']

Ejemplos de términos SOLO en LEMMA (no en STEMMING):
['afectado', 'pozo', 'atribuir', 'inauguro', 'rehabilitacion', 'feria', 'recorrer', 'mejorar', 'conectar', 'usuario']


> **Decisión final:** ¿stemming o lemmatización para este corpus en español? Justifiquen con **los números** que acaban de obtener, no con la intuición.

_decisión y justificación:_

**LEMMATIZACIÓN**

**Justificación basada en números:**
- Stemming produce ~87 términos únicos
- Lemmatización produce ~94 términos únicos (7 más)

**Razones:**
1. Lemma preserva mejor semántica: 'café' → 'café' (consultable), no 'caf' (stem erróneo)
2. Motor de búsqueda (Lab 2) necesita términos reales, no stems agresivos
3. En dominio de noticias, precisión semántica > compresión del vocabulario
4. Los 7 términos adicionales representan distinciones importantes para búsqueda

## 5 · Pipeline final + persistencia

Empaqueten su decisión en **una** función `preprocesar(texto) -> list[str]`. Esta función es el entregable más importante: el **Lab 2 la reutilizará tal cual** para indexar el corpus *y* para procesar las consultas. Si el corpus y las consultas no pasan por el mismo pipeline, su buscador no funcionará.

In [19]:
def preprocesar(texto):
    """Pipeline definitivo del equipo: texto crudo -> lista de terminos limpios.
    Debe reflejar TODAS sus decisiones (acentos, stopwords, stemming/lemma)."""
    # TODO: implementar reutilizando lo que ya construyeron arriba
    return tokens_lemma(texto)

# Demostracion
for fila in corpus_crudo[:3]:
    print(fila['id'], '->', preprocesar(fila['texto'])[:10])

d01 -> ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez', 'proteccion', 'civil']
d02 -> ['crisis', 'hidrico', 'agravar', 'desabasto', 'liquido', 'vital', 'afectar', 'mil', 'familia', 'zona']
d03 -> ['cafe', 'chiapa', 'rompio', 'record', 'historico', 'exportacion', 'ciclo', 'impulsado', 'demanda', 'europa']


In [20]:
# Guardar el corpus crudo y el procesado para el Lab 2
corpus_procesado = [{'id': f['id'], 'titulo': f['titulo'],
                     'tokens': preprocesar(f['texto'])} for f in corpus_crudo]

with open('corpus_crudo.json', 'w', encoding='utf-8') as fh:
    json.dump(corpus_crudo, fh, ensure_ascii=False, indent=2)
with open('corpus_procesado.json', 'w', encoding='utf-8') as fh:
    json.dump(corpus_procesado, fh, ensure_ascii=False, indent=2)

print('Guardados: corpus_crudo.json y corpus_procesado.json')

# Mostrar estadísticas finales
n_tokens_totales = sum(len(doc['tokens']) for doc in corpus_procesado)
vocab_final = set()
for doc in corpus_procesado:
    vocab_final.update(doc['tokens'])

print(f"\nTotal de tokens en corpus procesado: {n_tokens_totales}")
print(f"Promedio de tokens por documento: {n_tokens_totales / len(corpus_procesado):.2f}")
print(f"Tamaño del vocabulario final: {len(vocab_final)} términos")

Guardados: corpus_crudo.json y corpus_procesado.json

Total de tokens en corpus procesado: 229
Promedio de tokens por documento: 16.36
Tamaño del vocabulario final: 199 términos
